<a href="https://colab.research.google.com/github/manvento/master_geoai_roma3_2026/blob/main/Simulazione_di_un_Sistema_Agentico_per_la_Gestione_delle_Anomalie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulazione di un Sistema Agentico per la Gestione delle Anomalie

Questo notebook simula un sistema agentico basato su LLM (Large Language Models) e LangChain/LangGraph, progettato per monitorare e reagire automaticamente ad anomalie provenienti da eventi inviati da sistemi IoT collegati al contesto fisico.

L'obiettivo è dimostrare come agenti intelligenti possano collaborare per:

1.  **Rilevare e filtrare** log provenienti da sensori IoT.
2.  **Analizzare la gravità** delle anomalie identificate.
3.  **Innescare azioni appropriate** in base al livello di pericolo (ad esempio, inviare email urgenti o creare ordini di lavoro per la manutenzione).

![https://github.com/manvento/master_geoai_roma3_2026](https://github.com/manvento/master_geoai_roma3_2026/blob/main/docs/qr_code_geoai.png?raw=true)

## Framework e inizializzazione notebook

### Scelta del Framework: LangChain con Google GenAI

Per la realizzazione di questo sistema agentico, abbiamo optato per il framework **LangChain**, in particolare la sua integrazione ottimizzata con **Google GenAI**. Questa scelta è strategica per diverse ragioni:

*   **Integrazione Profonda**: LangChain offre un'integrazione nativa e fluida con i modelli di Large Language Model (LLM) di Google (come Gemini), rendendo semplice l'interazione e la gestione delle chiamate API.
*   **Compatibilità con Colab**: La versione per Google GenAI è la soluzione più integrata e performante all'interno dell'ambiente di Google Colab, garantendo un'esperienza di sviluppo ottimale.
*   **Flessibilità e Modularità**: LangChain permette di costruire facilmente catene di agenti e di gestire flussi di lavoro complessi, ideali per la nostra architettura multi-agente.

### Chiave API

La chiave API per provare questo notebook, va impostata all'interno Colab Secret. Ecco come fare:

1.  **Aggiungi il Secret**: Clicca sull'icona della chiave nel pannello a sinistra (o vai su `File > Apri il riquadro Secret`).
2.  **Crea un nuovo Secret**: Clicca su `+ Nuovo Secret`.
3.  **Nome del Secret**: Assegna il nome `GOOGLE_API_KEY` (deve corrispondere al nome usato nel codice).
4.  **Valore del Secret**: Incolla la tua Google API Key nel campo `Valore`.
5.  **Abilita per questo Notebook**: Assicurati che l'opzione `Notebook access` sia attiva per questo notebook nel pannello Secret.

In [ ]:
# Installa le librerie necessarie per questo progetto
%pip install -qU langchain-google-genai langgraph langchain-core

# Crea le directory per i dataset
!mkdir -p data

# Scarica i dataset da GitHub
!wget -O data/test_dataset_v1.csv https://raw.githubusercontent.com/manvento/master_geoai_roma3_2026/main/data/test_dataset_v1.csv
!wget -O data/test_dataset_v2.csv https://raw.githubusercontent.com/manvento/master_geoai_roma3_2026/main/data/test_dataset_v2.csv
!wget -O data/train_dataset.csv https://raw.githubusercontent.com/manvento/master_geoai_roma3_2026/main/data/train_dataset.csv

print("Datasets scaricati da GitHub!")

In [ ]:

# Importa il modulo `userdata` da google.colab
from google.colab import userdata
import os

# Recupera la chiave API dal Secret chiamato 'GOOGLE_API_KEY'
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY # Imposta la variabile d'ambiente
    print("Chiave API recuperata con successo da Colab Secrets e impostata come variabile d'ambiente!")
except userdata.SecretNotFoundError:
    print("Errore: Il Secret 'GOOGLE_API_KEY' non è stato trovato. Assicurati di averlo configurato correttamente nel pannello Secret di Colab.")
except Exception as e:
    print(f"Errore durante il recupero del Secret: {e}")

from typing import TypedDict, List, Dict, Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

# Inizializza il modello (Gemini 2.5 Flash è veloce e ottimo per questo scopo)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


# Struttura del record dell'evento riportato dall'IoT

from typing import TypedDict, Dict, Any, Optional

class IoTState(TypedDict, total=False):
    event_data: Dict[str, Any]
    anomalies_found: str
    danger_level: str
    confidence: float
    email_draft: Optional[str]
    work_order: Optional[str]

![Architettura Agentica v1](https://github.com/manvento/master_geoai_roma3_2026/blob/a7ce3e67d33c7f3be80da8653a79fcf848c81381/docs/architettura_agentica_v1.png?raw=true)


L'**Agente Collettore** è il primo agente nel nostro sistema. Il suo compito principale è quello di filtrare e processare i log grezzi provenienti dai dispositivi IoT. In pratica, questo agente:

*   **Riceve i `raw_logs`**: Prende in ingresso una stringa contenente i dati di log non elaborati.
*   **Interroga l'LLM**: Utilizza il modello di linguaggio (LLM) per analizzare il testo dei log e identificare eventuali anomalie presenti.
*   **Riassume le Anomalie**: Se vengono trovate anomalie, genera un riassunto conciso delle stesse. Se non ne trova, indica chiaramente "Nessuna anomalia".
*   **Aggiorna lo stato**: Restituisce un dizionario contenente la stringa `anomalies_found` che verrà passata agli agenti successivi nel grafo.

Questo agente funge da primo filtro, trasformando un flusso di dati grezzi in informazioni strutturate e rilevanti per l'analisi successiva.

In [ ]:
def agente_collettore(state: IoTState) -> dict:
    d = state["event_data"]

    summary = f"""
    - **Machine type**: {d.get('machine_type')}
    - **Sector**: {d.get('sector')}
    - **Shift**: {d.get('shift')}
    - **Temperature**: {d.get('temperature_c')} °C
    - **Vibration**: {d.get('vibration_mm_s')} mm/s
    - **Pressure**: {d.get('pressure_bar')} bar
    - **Oil level**: {d.get('oil_level_pct')} %
    - **Coolant level**: {d.get('coolant_level_pct')} %
    - **Humidity**: {d.get('humidity_pct')} %
    - **Voltage**: {d.get('voltage_v')} V
    - **Current**: {d.get('current_a')} A
    - **Power**: {d.get('power_kw')} kW
    - **RPM**: {d.get('rpm')}
    - **Noise**: {d.get('noise_db')} dB
    - **Smoke**: {d.get('smoke_ppm')} ppm
    - **Gas**: {d.get('gas_ppm')} ppm
    - **Network latency**: {d.get('network_latency_ms')} ms
    - **Errors (24h)**: {d.get('error_count_24h')}
    - **Restarts (7d)**: {d.get('restart_count_7d')}
    """.strip()

    return {"anomalies_found": summary}

L'**Agente A (Emergenza)** entra in azione quando l'**Agente Analizzatore** rileva un livello di pericolo **GRAVE**. Il suo compito è quello di gestire le situazioni di emergenza, concentrandosi sulla comunicazione rapida e formale. In particolare, questo agente:

*   **Riceve `anomalies_found`**: Prende in ingresso il riassunto delle anomalie gravi identificate dall'agente collettore.
*   **Genera un'email urgente**: Utilizza l'LLM per comporre un'email formale e urgente, indirizzata al Responsabile Impianto, descrivendo l'anomalia in modo chiaro e completo per richiedere un intervento immediato.
*   **Aggiorna lo stato**: Restituisce un dizionario contenente la stringa `email_draft` con il testo completo dell'email da inviare.

Questo agente assicura che le situazioni più critiche vengano escalate immediatamente attraverso canali di comunicazione appropriati.

In [ ]:
def agente_email_emergenza(state: IoTState) -> dict:

    print("[Agente A]: Pericolo GRAVE rilevato! Preparazione email d'emergenza...")
    prompt = f"""Scrivi un'email formale e urgente al Responsabile Impianto.
    Descrivi la seguente anomalia in modo chiaro e completo per un intervento immediato:
    {state['anomalies_found']}

    Oggetto dell'email: [URGENTE] Allarme Critico Impianto IoT.

    IMPORTANTE PER LA FIRMA: Per andare a capo correttamente nel formato Markdown,
    devi lasciare una riga vuota tra una riga e l'altra della firma (doppio invio).
    La firma deve apparire esattamente così:

    Mario Rossi
    Responsabile Servizio Manutenzione Predittiva
    email: mario.rossi@smartfactory.it"""

    start_time = time.time()

    response = llm.invoke([HumanMessage(content=prompt)])
    durata_ms = (time.time() - start_time) * 1000
    print(f"   > Tempo di elaborazione LLM: {durata_ms:.2f} ms")

    return {"email_draft": response.content}

L'**Agente B (Manutenzione)** si attiva quando l'**Agente Analizzatore** rileva un livello di pericolo che richiede **MANUTENZIONE**. Il suo scopo è quello di organizzare le azioni necessarie per la risoluzione del problema. Nello specifico, questo agente:

*   **Riceve `anomalies_found`**: Prende in ingresso il riassunto delle anomalie rilevate dall'agente collettore.
*   **Genera un Alert Mappa**: Utilizza l'LLM per estrarre informazioni rilevanti (es. ID sensore, luogo) e crea una mappa dove viene indicata la posizione dell'asset. Questo serve per simulare la visualizzazione del problema in una dashboard di controllo, che è un'attività tipica in cui viene utilizzato **NextGen Geo**.
*   **Crea un Work Order (Ordine di Lavoro)**: Compone uno scheletro di ordine di lavoro per i tecnici, basato sull'anomalia rilevata. Questo work order include dettagli come la descrizione del problema, parti di ricambio suggerite e una priorità (es. Media/Bassa).
*   **Aggiorna lo stato**: Restituisce un dizionario contenente `map_alert` (con le informazioni per la mappa) e `work_order` (il testo dell'ordine di lavoro).

Questo agente è cruciale per la gestione proattiva e organizzata delle problematiche che non richiedono un'emergenza immediata ma necessitano comunque di un intervento.

In [ ]:
def agente_work_order(state: IoTState) -> dict:

  prompt_wo = f"""Scrivi uno scheletro di Work Order per i tecnici basato su:
              {state['anomalies_found']}. Includi: ID Ticket, Descrizione problema,
              Parti di ricambio suggerite, Priorità (Media/Bassa), Numero di serie,
              tipologia di asset, Produttore, versione firmware, centro operativo,
              posizione nella rete, municipio.
              Non scrivere la tua introduzione (tipo ecco uno scheletro di work order...).
              Il nome dell'azienda è SmartFactory SpA.
              Non inserire parti da compilare, questo prompt contiene le
              informazioni identificative base per il work order. Cerca di essere sintetico."""

  start_time = time.time()

  response = llm.invoke([HumanMessage(content=prompt_wo)])
  durata_ms = (time.time() - start_time) * 1000
  print(f"   > Tempo di elaborazione LLM: {durata_ms:.2f} ms")


  return {"work_order": response.content}


## Agenti Non-LLM nel Sistema IoT

In un sistema agentico, **non è sempre necessario che tutti gli agenti siano Large Language Models (LLM)**. Usare un LLM per compiti semplici può essere come sparare con un cannone a una mosca.

L'agente incaricato di determinare e analizzare la gravità di una segnalazione può essere:

*   **Programmatico**: Basato su regole specifiche, termini chiave o codici predefiniti.
*   **Basato su Machine Learning tradizionale**: Addestrato per classificare gli eventi in base a dati storici.

Proviamo ora a simulare un agente analizzatore addestrato tramite Machine Learning per dimostrarne l'efficacia.

In [ ]:
# Tramite richiesta all'assisente Gemini,
# aggiungere la visualizzazione dei dataset data/train_dataset.csv e data/test_dataset_v1.csv



In [ ]:
# Partiamo con l'addestramento del modello di ML, basato su scikit-learn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer # Aggiunto l'import mancante
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

FEATURE_COLUMNS = None
ml_model = None

def train_iot_severity_model(csv_path: str):
    global FEATURE_COLUMNS, ml_model

    df = pd.read_csv(csv_path)

    if "severity" not in df.columns:
        raise ValueError("La colonna 'severity' non è presente nel dataset.")

    drop_cols = ["severity"]
    if "machine_id" in df.columns:
        drop_cols.append("machine_id")

    X = df.drop(columns=drop_cols)
    y = df["severity"]

    FEATURE_COLUMNS = X.columns.tolist()

    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_cols
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorical_cols
            )
        ],
        remainder="drop"
    )

    model = Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print("✅ Modello addestrato")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    ml_model = model
    return {"model": model, "accuracy": acc, "report": classification_report(y_test, y_pred, output_dict=False)}


In [ ]:
model_results = train_iot_severity_model("data/train_dataset.csv")
ml_model = model_results["model"]

# Output del modello addestrato
print("✅ Modello addestrato")
print(f"Accuracy: {model_results['accuracy']:.4f}")
print("\nClassification report:")
print(model_results['report'])

In [ ]:
# E costruiamo una funzione di predizione

def to_python_types(obj):
    """
    Converte ricorsivamente tipi numpy in tipi Python nativi,
    così LangGraph può serializzarli nel checkpointer.
    """
    if isinstance(obj, dict):
        return {k: to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_python_types(v) for v in obj]
    elif isinstance(obj, tuple):
        return tuple(to_python_types(v) for v in obj)
    elif isinstance(obj, np.generic):
        return obj.item()
    else:
        return obj

# Funzione di predizione

def preprocessing(event_dict: Dict[str, Any]) -> pd.DataFrame:
    if FEATURE_COLUMNS is None:
        raise ValueError("FEATURE_COLUMNS non inizializzate. Addestra prima il modello.")

    data = {col: event_dict.get(col, None) for col in FEATURE_COLUMNS}
    return pd.DataFrame([data])

def predict_iot_severity(model, event_dict: Dict[str, Any]):
    X = preprocessing(event_dict)

    pred = to_python_types(model.predict(X)[0])

    confidence = None
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)[0]
        confidence = dict(zip(model.classes_, probs))

    return pred, confidence


In [ ]:
#Infine l'agente che analizza la gravità del problema basandosi su quanto appreso tramite Machine Learning.

def agente_analizzatore(state: IoTState) -> dict:
    start_time = time.time()
    print("🤖 [ML] Analisi dati IoT...")

    event_data = state["event_data"]

    if not isinstance(event_data, dict):
        print("⚠️ Input non valido per il modello ML.")
        return {
            "danger_level": "WARNING",
            "requires_review": True
        }

    predizione, probabilita = predict_iot_severity(ml_model, event_data)

    durata_ms = (time.time() - start_time) * 1000

    print(f"   > Risultato ML: {predizione}")

    print(f"   > Tempo di elaborazione: {durata_ms:.2f} ms")

    return {
        "danger_level": str(predizione)
    }

Il **Router Condizionale** (implementato dalla funzione `router_condizionale`) è un componente cruciale del sistema che decide il flusso logico a seguito dell'analisi delle anomalie. Dopo che l'**Agente Analizzatore** ha determinato la `severity`, questo router entra in azione per instradare l'esecuzione al nodo appropriato:


["NORMAL", "MAINTENANCE", "WARNING", "FAILURE", "SEVERE"],
*   Se `severity` è **SEVERE**, il flusso viene indirizzato all'**Agente A (Emergenza)**.
*   Se `severity` è **FAILURE** o **WARNING**, il flusso viene indirizzato all'**Agente B (Manutenzione)**.
*   Se `severity` è **NORMAL** il processo termina (`END`).

Questo router garantisce che solo gli agenti necessari vengano attivati in base alla gravità dell'evento, ottimizzando l'efficienza del sistema e assicurando una risposta mirata.

Si tratta di un esempio di agente non basato su AI.

In [ ]:
def router_condizionale(state: IoTState) -> str:
    level = state.get("danger_level", "NORMAL")

    if level in ["SEVERE", "FAILURE"]:
        return "agente_a"

    if level in ["WARNING", "MAINTENANCE"]:
        return "agente_b"

    return "fine"

Ora procediamo con la **Costruzione del Grafo**, andando a definire l'intera architettura e il flusso logico del nostro sistema agentico.


In [ ]:
# Costruzione del Grafo
from langgraph.graph import StateGraph, END

# Inizializza il grafo con il nostro stato
workflow = StateGraph(IoTState)

# Aggiungiamo i nodi al grafo
workflow.add_node("collettore", agente_collettore)
workflow.add_node("analizzatore", agente_analizzatore)
workflow.add_node("agente_a", agente_email_emergenza)
workflow.add_node("agente_b", agente_work_order)

# Definiamo il flusso (gli archi)
workflow.set_entry_point("collettore")
workflow.add_edge("collettore", "analizzatore")

# Aggiungiamo l'arco condizionale (il router)
workflow.add_conditional_edges(
    "analizzatore",
    router_condizionale,
    {
        "agente_a": "agente_a",
        "agente_b": "agente_b",
        "fine": END
    }
)

# Definiamo la fine del processo per gli agenti A e B
workflow.add_edge("agente_a", END)
workflow.add_edge("agente_b", END)

# Compiliamo l'applicazione
iot_event_analyzer_v1 = workflow.compile()
print("Grafo compilato con successo!")

Scriviamo delle funzionalità di utilità che permettono di visualizzare gli eventi, aggiungendo un pin in mappa per gli allarmi che generano Work Order e allarmi collegati a email pronte per essere inviate.

In [ ]:
from typing import Tuple
import folium
import markdown
from IPython.display import display, HTML

# Lista globale dei punti
punti_mappa = []

mappa_factory = folium.Map(
    location=[45.4642, 9.1900],
    zoom_start=15,
    tiles="CartoDB positron"
)

punti_mappa = []

def aggiungi_punto_mappa(issue_type: str, content: str, location: tuple[float, float]):
    global punti_mappa

    color_map = {
        "SEVERE": "red",
        "FAILURE": "darkred",
        "WARNING": "orange",
        "MAINTENANCE": "blue",
        "NORMAL": "green",
        "GRAVE": "red",
        "MANUTENZIONE": "orange"
    }

    icon_map = {
        "SEVERE": "bell",
        "FAILURE": "remove-sign",
        "WARNING": "warning-sign",
        "MAINTENANCE": "wrench",
        "NORMAL": "ok-sign",
        "GRAVE": "bell",
        "MANUTENZIONE": "wrench"
    }

    color = color_map.get(issue_type, "cadetblue")
    icon_name = icon_map.get(issue_type, "info-sign")

    popup = None

    if content:

      formatted_content = markdown.markdown(content)

      html = folium.Html(f"""
      <div style="
          font-family: sans-serif;
          font-size: 12px;
          min-width: 450px;
          max-height: 400px;
          overflow-y: auto;
          overflow-x: hidden;
          padding-right: 8px;
      ">
          <h4 style="margin-top:0; color:{color};">{issue_type}</h4>
          {formatted_content}
      </div>
      """, script=True)

      popup = folium.Popup(html, max_width=500)


    folium.Marker(
        location=location,
        popup=popup,
        tooltip=f"Dettagli: {issue_type}",
        icon=folium.Icon(color=color, icon=icon_name)
    ).add_to(mappa_factory)

    punti_mappa.append(location)
def riposiziona_mappa_con_buffer(buffer_ratio: float = 0.05, min_buffer: float = 0.001):
    global punti_mappa, mappa_factory

    if not punti_mappa:
        return

    if len(punti_mappa) == 1:
        lat, lon = punti_mappa[0]
        bounds = [
            [lat - min_buffer, lon - min_buffer],
            [lat + min_buffer, lon + min_buffer]
        ]
        mappa_factory.fit_bounds(bounds)
        return

    lat_min = min(p[0] for p in punti_mappa)
    lat_max = max(p[0] for p in punti_mappa)
    lon_min = min(p[1] for p in punti_mappa)
    lon_max = max(p[1] for p in punti_mappa)

    lat_delta = lat_max - lat_min
    lon_delta = lon_max - lon_min

    lat_buffer = max(lat_delta * buffer_ratio, min_buffer)
    lon_buffer = max(lon_delta * buffer_ratio, min_buffer)

    bounds = [
        [lat_min - lat_buffer, lon_min - lon_buffer],
        [lat_max + lat_buffer, lon_max + lon_buffer]
    ]

    mappa_factory.fit_bounds(bounds)
def mostra_mappa(width=1280, height=700):
    riposiziona_mappa_con_buffer()

    html_map = mappa_factory._repr_html_()

    display(HTML(f"""
    <div style="
        width: {width}px;
        max-width: {width}px;
        height: {height}px;
        overflow: hidden;
        border: 1px solid #ccc;
    ">
        {html_map}
    </div>
    """))
def reset_mappa():
    global mappa_factory, punti_mappa
    punti_mappa = []
    mappa_factory = folium.Map(
        location=[45.4642, 9.1900],
        zoom_start=15,
        tiles="CartoDB positron"
    )

##Simulazione

Ora che il grafo è stato definito e compilato, siamo pronti per **Simulare gli Eventi IoT**. Questo passaggio dimostra come il sistema agentico reagisce a diversi scenari di log, invocando il grafo con input specifici e osservando i risultati. Vedremo come il sistema identifica le anomalie, determina il livello di pericolo e, di conseguenza, innesca le azioni appropriate tramite gli agenti A o B.

In [ ]:
import uuid
import time

def esegui_simulazione(event_data: Dict[str, Any]):
    print("🔍 Analizzando evento IoT tramite grafo agentico...")

    res = iot_event_analyzer_v1.invoke(
        {"event_data": event_data},
        config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    )

    print(f"✅ Severity classificata: {res.get('danger_level')}")

    lat = event_data.get("latitude")
    lon = event_data.get("longitude")

    if lat is None or lon is None:
        print("⚠️ Coordinate mancanti: impossibile mostrare il punto in mappa.")
        return res

    location = (lat, lon)

    if res.get("email_draft"):
        aggiungi_punto_mappa(res.get("danger_level", "SEVERE"), res["email_draft"], location)

    elif res.get("work_order"):
        aggiungi_punto_mappa(res.get("danger_level", "WARNING"), res["work_order"], location)

    else:
        # Evento normale, senza allarmi.

        aggiungi_punto_mappa(res.get("danger_level", "NORMAL"), "", location)

    return res

## Esecuzione vera e propria della simulazione

reset_mappa()

#leggiamo il dataset con le informazioni ricevute da IoT
df_test = pd.read_csv("data/test_dataset_v1.csv")

for _, row in df_test.iterrows():
    event_data = to_python_types(row.to_dict())
    esegui_simulazione(event_data)


mostra_mappa()

# Strategia di Sicurezza: Confidenza e Supervisione

Per rendere il sistema affidabile in un contesto critico, come quello del Field Service Managament, l’automazione non deve essere solo accurata, ma anche **controllabile**.

![Architettura Agentica v2](https://github.com/manvento/master_geoai_roma3_2026/blob/a7ce3e67d33c7f3be80da8653a79fcf848c81381/docs/architettura_agentica_v2.png?raw=true)

1. **Confidenza del modello**: L'agente ML restituirsce anche una distribuzione di probabilità che permetter di distingue *casi chiari* da *casi ambigui*
2. **Secondo parere**: Nei casi ambigui viene chiesta un secondo parere, nei casi meno critici ad un modello GenAI.
3. **Supervisione umana**: Per quelli più sensibili si adotta un approccio *human-in-the-loop*.


Questa strategia, oltre all'explainability delle decisioni automatiche, sono due concetti chiave per l'addozione di metodologie AI in modo conforme all'UE AI-Act e anche alle certificazioni di sicurezza che vanno garantite.

## Estrazione del livello di confidenza dal modello ML

In [ ]:
import time
import numpy as np

CONFIDENCE_THRESHOLD = 0.65

def _extract_confidence_score(confidence_value) -> float:
    """
    Converte il campo 'confidence' in uno score scalare.
    Supporta sia:
    - dict di probabilità per classe
    - float singolo
    """
    if isinstance(confidence_value, dict) and confidence_value:
        return float(max(confidence_value.values()))
    if isinstance(confidence_value, (int, float, np.floating)):
        return float(confidence_value)
    return 0.0


def analizzatore_ml_con_confidenza(state: IoTState) -> dict:
    """
    Usa il modello numerico già addestrato per classificare la severity
    e restituisce anche il livello di confidenza.
    """
    start_time = time.time()
    print("🤖 [ML] Analisi dati IoT con stima di confidenza...")

    event_data = state["event_data"]

    if not isinstance(event_data, dict):
        print("⚠️ Input non valido per il modello ML.")
        return {
            "danger_level": "WARNING",
            "confidence": {},
            "requires_review": True
        }

    predizione, probabilita = predict_iot_severity(ml_model, event_data)
    conf_score = _extract_confidence_score(probabilita)

    durata_ms = (time.time() - start_time) * 1000

    print(f"   > Risultato ML: {predizione}")
    print(f"   > Confidenza massima: {conf_score:.2%}")
    if probabilita:
        # Convert numpy.float64 values to standard Python floats for serialization
        probabilita_serializable = {k: float(v) for k, v in probabilita.items()}
        print(f"   > Distribuzione probabilità: {probabilita_serializable}")
    else:
        probabilita_serializable = {}
    print(f"   > Tempo di elaborazione: {durata_ms:.2f} ms")

    return {
        "danger_level": predizione,
        "confidence": probabilita_serializable,
        "requires_review": conf_score < CONFIDENCE_THRESHOLD
    }

## Nodo "Secondo Parere" e aggiornamento router

In [ ]:
def agente_secondo_parere(state: IoTState) -> dict:
    """
    Secondo parere sui casi a bassa confidenza.
    Qui manteniamo una logica semplice e trasparente:
    il nodo rilegge il riepilogo e, in uno scenario reale, potrebbe
    usare un LLM o una regola più sofisticata.
    """
    print("🧠 [Secondo Parere] Confidenza bassa. Rivalutazione dell'evento...")

    summary = state.get("anomalies_found", "")
    event_data = state.get("event_data", {})

    # Euristica semplice e leggibile, adatta alla demo
    smoke = event_data.get("smoke_ppm", 0) or 0
    gas = event_data.get("gas_ppm", 0) or 0
    temp = event_data.get("temperature_c", 0) or 0
    vib = event_data.get("vibration_mm_s", 0) or 0
    pressure = event_data.get("pressure_bar", 0) or 0
    oil = event_data.get("oil_level_pct", 100) or 100

    if smoke > 20 or gas > 15 or temp > 110:
        verdetto = "SEVERE"
    elif temp > 95 or vib > 4.0 or pressure > 15:
        verdetto = "FAILURE"
    elif temp > 85 or vib > 3.0 or oil < 25:
        verdetto = "WARNING"
    elif oil < 35:
        verdetto = "MAINTENANCE"
    else:
        verdetto = "NORMAL"

    print(f"🧠 [Secondo Parere] Verdetto: {verdetto}")
    if summary:
        print("   > Basato sul riepilogo evento e sulle soglie di controllo.")

    # Dopo il secondo parere consideriamo il caso risolto,
    # quindi non manteniamo il flag di review aperto.
    return {
        "danger_level": verdetto,
        "confidence": {"second_opinion": 1.0},
        "requires_review": False
    }


def router_con_fallback(state: IoTState) -> str:
    """
    Logica:
    1. se la confidenza ML è bassa -> secondo parere
    2. altrimenti:
       - SEVERE / FAILURE -> agente A
       - WARNING / MAINTENANCE -> agente B
       - NORMAL -> fine
    """
    conf_score = _extract_confidence_score(state.get("confidence", {}))

    if state.get("requires_review", False) or conf_score < CONFIDENCE_THRESHOLD:
        return "vai_a_secondo_parere"

    level = state.get("danger_level", "NORMAL")

    if level in ["SEVERE", "FAILURE"]:
        return "vai_ad_A"
    if level in ["WARNING", "MAINTENANCE"]:
        return "vai_a_B"
    return "fine"


def router_post_secondo_parere(state: IoTState) -> str:
    level = state.get("danger_level", "NORMAL")

    if level in ["SEVERE", "FAILURE"]:
        return "vai_ad_A"
    if level in ["WARNING", "MAINTENANCE"]:
        return "vai_a_B"
    return "fine"

## Breakpoint con richiesta utente e aggiornamento router

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END

memory = MemorySaver()

builder = StateGraph(IoTState)

builder.add_node("collettore", agente_collettore)
builder.add_node("analizzatore", analizzatore_ml_con_confidenza)
builder.add_node("secondo_parere", agente_secondo_parere)
builder.add_node("agente_a", agente_email_emergenza)
builder.add_node("agente_b", agente_work_order)

builder.set_entry_point("collettore")
builder.add_edge("collettore", "analizzatore")

builder.add_conditional_edges(
    "analizzatore",
    router_con_fallback,
    {
        "vai_a_secondo_parere": "secondo_parere",
        "vai_ad_A": "agente_a",
        "vai_a_B": "agente_b",
        "fine": END
    }
)

builder.add_conditional_edges(
    "secondo_parere",
    router_post_secondo_parere,
    {
        "vai_ad_A": "agente_a",
        "vai_a_B": "agente_b",
        "fine": END
    }
)

builder.add_edge("agente_a", END)
builder.add_edge("agente_b", END)

# Punto di supervisione prima dell'azione critica
iot_event_analyzer_v2 = builder.compile(
    checkpointer=memory,
    interrupt_before=["agente_a"]
)

print("✅ Grafo avanzato con fallback e supervisione compilato.")

## Simulazione con supervisione

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END

memory = MemorySaver()

builder = StateGraph(IoTState)

builder.add_node("collettore", agente_collettore)
builder.add_node("analizzatore", analizzatore_ml_con_confidenza)
builder.add_node("secondo_parere", agente_secondo_parere)
builder.add_node("agente_a", agente_email_emergenza)
builder.add_node("agente_b", agente_work_order)

builder.set_entry_point("collettore")
builder.add_edge("collettore", "analizzatore")

builder.add_conditional_edges(
    "analizzatore",
    router_con_fallback,
    {
        "vai_a_secondo_parere": "secondo_parere",
        "vai_ad_A": "agente_a",
        "vai_a_B": "agente_b",
        "fine": END
    }
)

builder.add_conditional_edges(
    "secondo_parere",
    router_post_secondo_parere,
    {
        "vai_ad_A": "agente_a",
        "vai_a_B": "agente_b",
        "fine": END
    }
)

builder.add_edge("agente_a", END)
builder.add_edge("agente_b", END)

# Punto di supervisione prima dell'azione critica
iot_event_analyzer_v2 = builder.compile(
    checkpointer=memory,
    interrupt_before=["agente_a"]
)

print("✅ Grafo avanzato con fallback e supervisione compilato.")

## Esecuzione simulazione con strategia a tre step di controllo

In [ ]:
import uuid

def esegui_simulazione_avanzata(event_data: Dict[str, Any], auto_approve: Optional[bool] = None):
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    print("\n📥 EVENTO IoT IN INGRESSO:")
    print(event_data)

    print("\n--- Avvio workflow ---")
    for step in iot_event_analyzer_v2.stream({"event_data": event_data}, config):
        print("STEP:", step)

    snapshot = iot_event_analyzer_v2.get_state(config)

    print("\n--- Snapshot dopo il primo run ---")
    print("next:", snapshot.next)
    print("values:", snapshot.values)

    if snapshot.next:

        print("\n🛑 ***[SISTEMA IN PAUSA] Azione critica pronta, richiesta approvazione. ***")
        print(f"Livello rilevato: {snapshot.values.get('danger_level')}")
        print(f"Confidenza: {snapshot.values.get('confidence')}")

        if auto_approve is None:
          scelta = input("Confermi l'esecuzione dell'azione critica? (s/n): ").strip().lower()
          approved = scelta == "s"
        else:
          approved = auto_approve
          print(f"👤 Decisione operatore simulata: {'APPROVA' if approved else 'RIFIUTA'}")

        if approved:
            print("🚀 Azione approvata dall'operatore.")
            for step in iot_event_analyzer_v2.stream(None, config):
                print("RESUME STEP:", step)
        else:
            print("❌ Azione annullata dall'operatore.")
            return snapshot.values

    stato_finale = iot_event_analyzer_v2.get_state(config).values

    lat = event_data.get("latitude")
    lon = event_data.get("longitude")
    location = (lat, lon) if lat is not None and lon is not None else None

    if "email_draft" in stato_finale and stato_finale["email_draft"]:
        print("\n📧 Risultato finale: email di emergenza preparata.")
        if location:
            aggiungi_punto_mappa(
                stato_finale.get("danger_level", "SEVERE"),
                stato_finale["email_draft"],
                location
            )

    elif "work_order" in stato_finale and stato_finale["work_order"]:
        print("\n🔧 Risultato finale: work order generato.")
        if location:
            aggiungi_punto_mappa(
                stato_finale.get("danger_level", "WARNING"),
                stato_finale["work_order"],
                location
            )

    else:
        print("\nℹ️ Risultato finale: nessuna escalation operativa.")

    return stato_finale

In [ ]:
reset_mappa()

#leggiamo il dataset con le informazioni ricevute da IoT
# Il dataset test_dataset_v2.csv è già stato scaricato nella cartella 'data'
# dal blocco di codice iniziale.
df_test = pd.read_csv("data/test_dataset_v2.csv")

for _, row in df_test.iterrows():
    event_data = to_python_types(row.to_dict())
    esegui_simulazione_avanzata(event_data)


mostra_mappa()